# Build the upxBuilder Android APK on Google Colab — signed RELEASE (R8 + obfuscation)

No PC setup needed — Colab provides the Linux machine; Java and the Android SDK are installed below. The release build is shrunk and obfuscated by **R8** (`minifyEnabled` + `shrinkResources` + ProGuard rules).

**How to use:** menu **Runtime → Run all**, wait ~10–15 minutes, and the download cell hands you `app-release.apk`.

**Signing (important):** the cell after setup stores a release keystore in **your Google Drive** and reuses it on every build, so each new APK installs cleanly *over* the previous one. If you skip it, a one-session key is used instead — then updating across sessions requires uninstalling the app first (which deletes its installed toolchains).

> ⚠️ Switching from the old **debug** APK to this **release** APK changes the signature once: if Android says "App not installed", uninstall the old version first, then install this one.

In [ ]:
%%bash
# ── Step 1: Java 17 + Android SDK ──────────────────────────────────────
set -e
echo '==> Installing Java 17…'
apt-get -qq update >/dev/null
apt-get -qq install -y openjdk-17-jdk-headless >/dev/null

echo '==> Downloading Android command-line tools…'
mkdir -p /content/android-sdk/cmdline-tools
cd /content/android-sdk/cmdline-tools
rm -rf latest cmdline-tools tools.zip
wget -q -O tools.zip https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip
unzip -q tools.zip && rm tools.zip
mv cmdline-tools latest

echo '==> Installing SDK components (platform 34, build-tools, NDK 26.1, CMake 3.22)…'
yes | latest/bin/sdkmanager --licenses >/dev/null 2>&1 || true
latest/bin/sdkmanager --install 'platform-tools' 'platforms;android-34' 'build-tools;34.0.0' 'ndk;26.1.10909125' 'cmake;3.22.1' >/dev/null
echo 'Setup done.'

In [ ]:
# ── Step 2: release signing key, kept in YOUR Google Drive ─────────────
# Creates the keystore once and reuses it on every later build, so each new
# APK installs over the previous one. Approve the Drive permission popup.
from google.colab import drive
drive.mount('/content/drive')

import os, secrets, pathlib
keydir = pathlib.Path('/content/drive/MyDrive/upxBuilder-keys')
keydir.mkdir(parents=True, exist_ok=True)
ks, pw_file = keydir / 'upx-release.jks', keydir / 'keystore-password.txt'

if not pw_file.exists():
    pw_file.write_text(secrets.token_urlsafe(16))
pw = pw_file.read_text().strip()

if not ks.exists():
    rc = os.system(
        f'/usr/lib/jvm/java-17-openjdk-amd64/bin/keytool -genkeypair -v '
        f'-keystore "{ks}" -alias upx -keyalg RSA -keysize 2048 -validity 10000 '
        f'-storepass "{pw}" -keypass "{pw}" '
        f'-dname "CN=upxBuilder,O=upxBuilder,C=US" > /dev/null 2>&1')
    print('Created a new release keystore in Google Drive.' if rc == 0
          else 'keytool failed — make sure the Step 1 cell ran first.')
else:
    print('Reusing the existing release keystore from Google Drive.')

with open('/content/signing.env', 'w') as f:
    f.write(f'export UPX_KEYSTORE="{ks}"\n'
            f'export UPX_KEYSTORE_PASSWORD="{pw}"\n'
            f'export UPX_KEY_ALIAS=upx\n'
            f'export UPX_KEY_PASSWORD="{pw}"\n')
print('Signing configured. Keep the upxBuilder-keys folder safe in your Drive.')

In [ ]:
%%bash
# ── Step 3: clone the repo and build the signed RELEASE APK (R8) ────────
set -e
export ANDROID_HOME=/content/android-sdk
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64

cd /content
rm -rf New-work
echo '==> Cloning DanyFox12/New-work…'
git clone -q https://github.com/DanyFox12/New-work.git
cd New-work
git checkout -q claude/practical-faraday-fd4wyi

cd upxBuilder-android
echo "sdk.dir=$ANDROID_HOME" > local.properties
chmod +x gradlew

# Bundle the whole Linux environment inside the APK (AndroidIDE-style):
#  - the Android-native proot launcher (green-green-avk/build-proot-android)
#  - the Alpine Linux base system (~4 MB per CPU)
# The app then sets up its terminal environment with ZERO network. If this
# step is skipped, the app downloads the same files at install time instead.
echo '==> Bundling the Android proot launcher into the APK assets…'
mkdir -p app/src/main/assets/proot app/src/main/assets/alpine
for arch in aarch64 armv7a x86_64 i686; do
  wget -q -O app/src/main/assets/proot/proot-android-$arch.tar.gz \
    https://raw.githubusercontent.com/green-green-avk/build-proot-android/master/packages/proot-android-$arch.tar.gz
done
echo '==> Bundling the Alpine Linux base system into the APK assets…'
for arch in aarch64 armv7 x86_64; do
  wget -q -O app/src/main/assets/alpine/alpine-minirootfs-$arch.tar.gz \
    https://dl-cdn.alpinelinux.org/alpine/v3.20/releases/$arch/alpine-minirootfs-3.20.3-$arch.tar.gz
done
ls -l app/src/main/assets/proot/ app/src/main/assets/alpine/

# Signing: use the persistent Drive keystore from Step 2 if it ran; otherwise
# fall back to a one-session key (updates across sessions then need uninstall).
if [ -f /content/signing.env ]; then
  source /content/signing.env
  echo '==> Signing with the persistent keystore from Google Drive.'
else
  export UPX_KEYSTORE=/content/upx-release.jks
  export UPX_KEYSTORE_PASSWORD=upxbuilder
  export UPX_KEY_ALIAS=upx
  export UPX_KEY_PASSWORD=upxbuilder
  if [ ! -f "$UPX_KEYSTORE" ]; then
    "$JAVA_HOME/bin/keytool" -genkeypair -v -keystore "$UPX_KEYSTORE" \
      -alias "$UPX_KEY_ALIAS" -keyalg RSA -keysize 2048 -validity 10000 \
      -storepass "$UPX_KEYSTORE_PASSWORD" -keypass "$UPX_KEY_PASSWORD" \
      -dname "CN=upxBuilder,O=upxBuilder,C=US" >/dev/null 2>&1
  fi
  echo '==> WARNING: one-session signing key (Step 2 was skipped).'
fi

echo '==> Building release with R8 (shrink + obfuscate) — be patient…'
./gradlew --no-daemon assembleRelease
echo
echo '==> Signed release APK ready:'
ls -lh app/build/outputs/apk/release/app-release.apk

In [ ]:
# ── Step 4: download the APK to this computer ──────────────────────────
from google.colab import files
files.download('/content/New-work/upxBuilder-android/app/build/outputs/apk/release/app-release.apk')

### Optional: save to Google Drive instead

Easier when you build from a phone browser — the APK appears in the Google Drive app on your phone; tap it there to install.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp /content/New-work/upxBuilder-android/app/build/outputs/apk/release/app-release.apk /content/drive/MyDrive/upxBuilder.apk
print('Saved to Drive as upxBuilder.apk — open the Google Drive app on your phone and tap it to install.')

## After installing on the phone

> If Android says **"App not installed"**, the new APK's signature differs from the installed one (e.g. old debug build, or a build signed with a different session key). Uninstall the old app once, then install this APK — afterwards, updates built with your Drive keystore install cleanly.

1. Open **upxBuilder** (phone needs internet). On first launch the app **offers to download the dev environment automatically** — pick *Essentials* (~6 MB) or *Everything* (~600 MB, Wi-Fi).
2. Or do it any time from **Setup** (download icon in the toolbar / welcome screen).
3. Add big SDKs when needed, from Setup or the Terminal tab:
   ```
   pkg install flutter           # Flutter + Dart (~1 GB, use Wi-Fi)
   pkg install sdk               # Android sdkmanager
   pkg install platform-tools    # adb & fastboot
   pkg help                      # everything available
   ```
4. Create a project (New Project → Flutter / C++ / Java / Kotlin / Python / JS / Go) and press **Run**.